# 10.4 怎麼保留圖片的空間位置？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**把圖左右交換和只換tensor儲存順序是同一件事嗎？

相同內容patch在不同位置，應帶不同空間線索。加入patch位置向量；重排內容時是否同時重排位置，代表不同操作。

**把直覺寫成數學：**x_i=patch_embedding_i+position_i；不固定空間對應就丟了位置。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
patches = torch.ones(1, 4, 3)
position = torch.arange(4.0)[None, :, None]
x = patches + position
print(x)
assert not torch.equal(x[:, 0], x[:, 3])

**如何讀結果：**這裡用數值示範位置，不是已學的視覺語意；encoder實作中位置是可學參數。

**只改一件事：**只移除position，檢查相同patch完全相同。
